# Module 7: Granger Causality Feature Ranking

## Learning Objectives

By completing this notebook, you will:
1. Compute Granger causality p-values for all candidate features against a financial target
2. Apply multiple testing correction (Bonferroni and BH-FDR) and understand the difference
3. Compare Granger ranking against correlation and mutual information rankings
4. Visualise a directed information flow network showing which features Granger-cause the target

## Prerequisites
- Module 07 Guide 01: Granger causality concepts and F-test
- Stationarity testing (ADF test)
- Basic familiarity with statsmodels

## Estimated Time: 15 minutes

## Dataset
We use the **FRED-MD** macroeconomic dataset — a publicly available monthly macroeconomic database
maintained by the Federal Reserve Bank of St. Louis. This gives us 128 macroeconomic and
financial variables to test as features for predicting industrial production growth.

Reference: McCracken, M.W. & Ng, S. (2016). FRED-MD: A Monthly Database for Macroeconomic Research.
*Journal of Business & Economic Statistics*, 34(4), 574-589.

## 1. Setup and Data Loading

In [ ]:
# Core numerical and data libraries
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Statistical tests
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
from statsmodels.stats.multitest import multipletests
from sklearn.feature_selection import mutual_info_regression
from scipy.stats import spearmanr

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# Network graph for directed information flow
import networkx as nx

# Style configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

print('Libraries loaded.')

### 1.1 Load FRED-MD Data

FRED-MD is available as a CSV download from the St. Louis Fed. We download and apply the
recommended transformations (first differences, log differences) to make variables stationary.

In [ ]:
# Purpose: Load real macroeconomic time series data
# FRED-MD: 128 monthly macro/financial variables, 1959-present
# Transformation codes from McCracken & Ng (2016):
#   1=level, 2=first diff, 3=second diff,
#   4=log, 5=log first diff, 6=log second diff, 7=% change in growth rate

FREDMD_URL = 'https://files.stlouisfed.org/files/htdocs/fred-md/monthly/current.csv'

try:
    raw = pd.read_csv(FREDMD_URL, index_col=0)
    # First row contains transformation codes
    transform_codes = raw.iloc[0].astype(float)
    data_raw = raw.iloc[1:].copy()
    data_raw.index = pd.to_datetime(data_raw.index)
    data_raw = data_raw.apply(pd.to_numeric, errors='coerce')
    print(f'FRED-MD loaded: {data_raw.shape[0]} months x {data_raw.shape[1]} variables')
    print(f'Date range: {data_raw.index[0].date()} to {data_raw.index[-1].date()}')
    data_source = 'fred_md'
except Exception as e:
    print(f'FRED-MD download failed ({e}). Generating synthetic macro data.')
    # Synthetic fallback: AR(1) processes with varying autocorrelation
    n_periods = 400
    n_features = 30
    dates = pd.date_range('1990-01-01', periods=n_periods, freq='ME')
    rng = np.random.default_rng(42)

    # Create target: industrial production growth with AR(1) dynamics
    ip_growth = np.zeros(n_periods)
    ip_growth[0] = rng.normal(0, 0.5)
    for t in range(1, n_periods):
        ip_growth[t] = 0.3 * ip_growth[t-1] + rng.normal(0, 0.5)

    # Create features: some Granger-cause target, others don't
    features_arr = np.zeros((n_periods, n_features))
    n_causal = 8
    for j in range(n_features):
        phi = rng.uniform(0.2, 0.7)
        features_arr[0, j] = rng.normal(0, 1)
        for t in range(1, n_periods):
            features_arr[t, j] = phi * features_arr[t-1, j] + rng.normal(0, 1)
            if j < n_causal:  # First n_causal features truly predict target (lag 1)
                features_arr[t, j] += 0.3 * ip_growth[t-1]

    feature_names = [f'MacroVar_{j+1:02d}' for j in range(n_features)]
    data_raw = pd.DataFrame(features_arr, index=dates, columns=feature_names)
    data_raw['INDPRO'] = ip_growth  # industrial production as target
    transform_codes = pd.Series({col: 1 for col in data_raw.columns})  # all already stationary
    data_source = 'synthetic'
    print(f'Synthetic data generated: {data_raw.shape}')

### 1.2 Apply Transformations and Select Variables

We apply the McCracken & Ng (2016) transformations to achieve stationarity, then select
a manageable subset of 30 candidate features to test against the industrial production growth target.

In [ ]:
# Purpose: Transform variables to stationarity per FRED-MD recommended codes

def apply_fredmd_transformation(series: pd.Series, code: float) -> pd.Series:
    """Apply FRED-MD transformation code to achieve stationarity."""
    s = series.copy()
    if code == 1:
        return s  # Level (already stationary)
    elif code == 2:
        return s.diff()  # First difference
    elif code == 3:
        return s.diff().diff()  # Second difference
    elif code == 4:
        return np.log(s.clip(lower=1e-10))  # Log
    elif code == 5:
        return np.log(s.clip(lower=1e-10)).diff()  # Log first difference
    elif code == 6:
        return np.log(s.clip(lower=1e-10)).diff().diff()  # Log second difference
    elif code == 7:
        return s.pct_change().diff()  # % change in growth
    return s

if data_source == 'fred_md':
    # Apply transformations
    transformed = pd.DataFrame(index=data_raw.index)
    for col in data_raw.columns:
        code = transform_codes.get(col, 1)
        transformed[col] = apply_fredmd_transformation(data_raw[col], code)

    # Use industrial production (INDPRO) as target — log first diff = monthly growth rate
    if 'INDPRO' in transformed.columns:
        target = transformed['INDPRO'].dropna()
        # Select 30 features across categories: financial, real activity, prices, labour
        candidate_cols = [
            c for c in transformed.columns
            if c != 'INDPRO'
            and transformed[c].dropna().std() > 0
            and len(transformed[c].dropna()) > 100
        ][:30]
        features = transformed[candidate_cols]
    else:
        target = transformed.iloc[:, 0]
        features = transformed.iloc[:, 1:31]
else:
    target = data_raw['INDPRO']
    features = data_raw[[c for c in data_raw.columns if c != 'INDPRO']]

# Align target and features
common_idx = target.dropna().index.intersection(features.dropna(how='all').index)
target = target.loc[common_idx].dropna()
features = features.loc[target.index].fillna(method='ffill').dropna()
target = target.loc[features.index]

print(f'Target: {target.name} ({len(target)} observations)')
print(f'Candidate features: {features.shape[1]}')
print(f'Date range: {target.index[0].date()} to {target.index[-1].date()}')
print(f'\nTarget statistics:')
print(target.describe().round(4))

---

## 2. Stationarity Verification

Before running Granger tests, verify that both the target and features are stationary.
Non-stationary series produce invalid F-test distributions.

In [ ]:
# Purpose: ADF test for stationarity on target and all features
# Key: Granger causality is only valid for stationary series

def adf_stationarity_check(series: pd.Series, name: str = '') -> dict:
    """Augmented Dickey-Fuller test for stationarity."""
    clean = series.dropna()
    if len(clean) < 20:
        return {'name': name, 'adf_stat': np.nan, 'pvalue': np.nan, 'stationary': False}
    adf_stat, p_value, _, _, crit, _ = adfuller(clean, autolag='AIC')
    return {
        'name': name,
        'adf_stat': adf_stat,
        'pvalue': p_value,
        'stationary': p_value < 0.05,
        'crit_5pct': crit['5%'],
    }

# Check target
target_check = adf_stationarity_check(target, target.name)
print(f"Target ({target.name}):")
print(f"  ADF stat: {target_check['adf_stat']:.3f}  |  p-value: {target_check['pvalue']:.4f}")
print(f"  Stationary: {target_check['stationary']}")

# Check features
print(f"\nFeature stationarity (ADF test, alpha=0.05):")
stationary_features = []
nonstationary_features = []

for col in features.columns:
    check = adf_stationarity_check(features[col], col)
    if check['stationary']:
        stationary_features.append(col)
    else:
        nonstationary_features.append(col)

print(f"  Stationary (ADF rejects unit root): {len(stationary_features)}")
print(f"  Non-stationary: {len(nonstationary_features)}")

if nonstationary_features:
    print(f"  Differencing non-stationary features: {nonstationary_features[:5]}...")
    for col in nonstationary_features:
        features[col] = features[col].diff()

# Drop NaN rows after potential differencing
features = features.dropna()
target = target.loc[features.index]

print(f"\nFinal dataset: {len(target)} observations, {features.shape[1]} features")

---

## 3. Granger Causality Testing

Run bivariate Granger causality tests for every feature against the target.
We test lag orders 1 through 6 and record the minimum p-value (the lag that shows
the strongest evidence of predictive causality).

In [ ]:
# Purpose: Compute bivariate Granger causality for all features vs target
# Key insight: min p-value across lags captures the best predictive lag

MAX_LAG = 6  # Test lags 1 through 6

def granger_test_feature(target: pd.Series, feature: pd.Series,
                          max_lag: int = 6) -> dict:
    """Bivariate Granger causality test for a single feature."""
    data = pd.concat([target.rename('y'), feature.rename('x')], axis=1).dropna()

    if len(data) < max_lag * 4 + 10:  # need enough observations
        return {'min_pvalue': 1.0, 'best_lag': 1, 'f_stat': 0.0, 'n_obs': len(data)}

    try:
        results = grangercausalitytests(data[['y', 'x']], maxlag=max_lag, verbose=False)
        pvalues = {lag: res[0]['ssr_ftest'][1] for lag, res in results.items()}
        f_stats = {lag: res[0]['ssr_ftest'][0] for lag, res in results.items()}
        best_lag = min(pvalues, key=pvalues.get)
        return {
            'min_pvalue': pvalues[best_lag],
            'best_lag': best_lag,
            'f_stat': f_stats[best_lag],
            'pvalues_by_lag': pvalues,
            'n_obs': len(data),
        }
    except Exception as e:
        return {'min_pvalue': 1.0, 'best_lag': 1, 'f_stat': 0.0, 'n_obs': len(data)}

print(f'Running Granger tests for {features.shape[1]} features (max lag = {MAX_LAG})...')
granger_records = []

for col in features.columns:
    result = granger_test_feature(target, features[col], MAX_LAG)
    granger_records.append({
        'feature': col,
        'granger_pvalue': result['min_pvalue'],
        'best_lag': result['best_lag'],
        'f_stat': result['f_stat'],
        'n_obs': result['n_obs'],
    })

granger_df = pd.DataFrame(granger_records).sort_values('granger_pvalue').reset_index(drop=True)

print(f'\nTop 10 features by Granger causality p-value:')
print(granger_df.head(10)[['feature', 'granger_pvalue', 'best_lag', 'f_stat']].to_string(index=False))

---

## 4. Multiple Testing Correction

With 30 features tested simultaneously at alpha=0.05, we expect ~1.5 false positives
under the null. Bonferroni controls the familywise error rate (FWER) but is conservative.
Benjamini-Hochberg FDR allows some false positives but retains more true positives.

In [ ]:
# Purpose: Apply multiple testing correction to control false discovery rate
# Key: BH-FDR is appropriate for feature selection (some false positives acceptable)

ALPHA = 0.05
pvalues_array = granger_df['granger_pvalue'].values

# Bonferroni correction
reject_bonf, pvals_bonf, _, _ = multipletests(pvalues_array, alpha=ALPHA, method='bonferroni')

# Benjamini-Hochberg FDR correction
reject_bh, pvals_bh, _, _ = multipletests(pvalues_array, alpha=ALPHA, method='fdr_bh')

granger_df['pvalue_bonferroni'] = pvals_bonf
granger_df['pvalue_bh_fdr'] = pvals_bh
granger_df['selected_bonferroni'] = reject_bonf
granger_df['selected_bh_fdr'] = reject_bh

n_bonf = reject_bonf.sum()
n_bh = reject_bh.sum()

print('Multiple Testing Correction Results')
print('='*60)
print(f'Features tested: {len(pvalues_array)}')
print(f'Alpha: {ALPHA}')
print(f'Expected false positives (no correction): {ALPHA * len(pvalues_array):.1f}')
print(f'Selected by Bonferroni (FWER control): {n_bonf}')
print(f'Selected by BH-FDR (FDR control): {n_bh}')

# Visualise p-value distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Raw p-values
axes[0].bar(range(len(pvalues_array)), sorted(pvalues_array), color='steelblue', alpha=0.7)
axes[0].axhline(ALPHA, color='red', linestyle='--', label=f'alpha={ALPHA}')
axes[0].set_title('Raw Granger p-values (sorted)', fontsize=12)
axes[0].set_xlabel('Feature rank')
axes[0].set_ylabel('p-value')
axes[0].legend()

# BH-FDR: p-value plot (sorted p-values vs BH threshold line)
sorted_p = np.sort(pvalues_array)
bh_thresholds = ALPHA * np.arange(1, len(sorted_p)+1) / len(sorted_p)
axes[1].plot(range(len(sorted_p)), sorted_p, 'o-', color='steelblue', markersize=4, label='Sorted p-values')
axes[1].plot(range(len(bh_thresholds)), bh_thresholds, '--', color='red', label='BH threshold')
axes[1].axhline(ALPHA / len(sorted_p), color='orange', linestyle=':', label='Bonferroni threshold')
axes[1].set_title('BH-FDR Procedure', fontsize=12)
axes[1].set_xlabel('Rank k')
axes[1].set_ylabel('p-value')
axes[1].legend(fontsize=8)

# Comparison: number selected
counts = [len(pvalues_array), n_bh, n_bonf]
labels = ['All tested', 'BH-FDR\nselected', 'Bonferroni\nselected']
colors = ['lightgray', 'steelblue', 'coral']
axes[2].bar(labels, counts, color=colors, edgecolor='black', linewidth=0.5)
axes[2].set_title('Features Selected by Method', fontsize=12)
axes[2].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[2].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('granger_pvalue_correction.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nFigure saved: granger_pvalue_correction.png')

---

## 5. Compare Granger vs Correlation vs Mutual Information Rankings

Granger causality captures temporal predictive precedence. Correlation captures contemporaneous
linear association. Mutual information captures any association (including nonlinear) but
ignores temporal ordering. Comparing these rankings reveals which features pass different
notions of relevance.

In [ ]:
# Purpose: Build comparison table of three feature ranking methods
# Key insight: features that rank highly in all three methods are most reliable

# Lagged correlation: correlate each feature's lag-1 with target (one-step ahead prediction)
lag1_corr = {}
for col in features.columns:
    feat_lag1 = features[col].shift(1).dropna()
    tgt_aligned = target.loc[feat_lag1.index].dropna()
    feat_aligned = feat_lag1.loc[tgt_aligned.index]
    if len(feat_aligned) > 10:
        lag1_corr[col] = abs(feat_aligned.corr(tgt_aligned))
    else:
        lag1_corr[col] = 0.0

# Mutual information with lag-1 features
feat_lag1_df = features.shift(1).dropna()
tgt_aligned = target.loc[feat_lag1_df.index].dropna()
feat_lag1_df = feat_lag1_df.loc[tgt_aligned.index].fillna(0)

mi_scores = mutual_info_regression(
    feat_lag1_df.values, tgt_aligned.values, random_state=42
)
mi_dict = dict(zip(features.columns, mi_scores))

# Build comparison DataFrame
comparison = granger_df[['feature', 'granger_pvalue', 'selected_bh_fdr']].copy()
comparison['granger_rank'] = range(1, len(comparison) + 1)  # already sorted by p-value

comparison['lag1_abs_corr'] = comparison['feature'].map(lag1_corr)
comparison['mi_score'] = comparison['feature'].map(mi_dict)

# Rank by each method
comparison['corr_rank'] = comparison['lag1_abs_corr'].rank(ascending=False).astype(int)
comparison['mi_rank'] = comparison['mi_score'].rank(ascending=False).astype(int)

# Compute rank differences
comparison['granger_vs_corr_rankdiff'] = abs(comparison['granger_rank'] - comparison['corr_rank'])
comparison['granger_vs_mi_rankdiff'] = abs(comparison['granger_rank'] - comparison['mi_rank'])

# Spearman rank correlation between methods
rho_granger_corr, _ = spearmanr(comparison['granger_rank'], comparison['corr_rank'])
rho_granger_mi, _ = spearmanr(comparison['granger_rank'], comparison['mi_rank'])
rho_corr_mi, _ = spearmanr(comparison['corr_rank'], comparison['mi_rank'])

print('Rank Correlation Between Feature Selection Methods')
print('='*50)
print(f'Granger vs Lag-1 Correlation: rho = {rho_granger_corr:.3f}')
print(f'Granger vs Mutual Information: rho = {rho_granger_mi:.3f}')
print(f'Correlation vs Mutual Info:    rho = {rho_corr_mi:.3f}')

print(f'\nTop 10 features (by Granger ranking) with all method ranks:')
print(comparison.head(10)[['feature', 'granger_rank', 'corr_rank', 'mi_rank',
                             'granger_pvalue', 'lag1_abs_corr', 'mi_score']].to_string(index=False))

In [ ]:
# Purpose: Visualise ranking disagreements across methods
# Key: large disagreements reveal features that predict differently depending on criterion

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Scatter: Granger rank vs Correlation rank
sc1 = axes[0].scatter(
    comparison['granger_rank'], comparison['corr_rank'],
    c=comparison['granger_pvalue'], cmap='RdYlGn_r',
    s=60, alpha=0.8, edgecolors='black', linewidths=0.5
)
axes[0].plot([1, len(comparison)], [1, len(comparison)], 'k--', alpha=0.3, label='Perfect agreement')
axes[0].set_xlabel('Granger Rank', fontsize=11)
axes[0].set_ylabel('Lag-1 Correlation Rank', fontsize=11)
axes[0].set_title(f'Granger vs Correlation\n(Spearman rho={rho_granger_corr:.2f})', fontsize=11)
plt.colorbar(sc1, ax=axes[0], label='Granger p-value')

# Scatter: Granger rank vs MI rank
sc2 = axes[1].scatter(
    comparison['granger_rank'], comparison['mi_rank'],
    c=comparison['mi_score'], cmap='YlOrRd',
    s=60, alpha=0.8, edgecolors='black', linewidths=0.5
)
axes[1].plot([1, len(comparison)], [1, len(comparison)], 'k--', alpha=0.3)
axes[1].set_xlabel('Granger Rank', fontsize=11)
axes[1].set_ylabel('Mutual Information Rank', fontsize=11)
axes[1].set_title(f'Granger vs Mutual Info\n(Spearman rho={rho_granger_mi:.2f})', fontsize=11)
plt.colorbar(sc2, ax=axes[1], label='MI score')

# Heatmap: agreement matrix (top 15 features by Granger)
top15 = comparison.head(15)
rank_matrix = top15[['granger_rank', 'corr_rank', 'mi_rank']].values
# Normalize ranks to [0, 1] for colour comparison
rank_norm = (rank_matrix - rank_matrix.min()) / (rank_matrix.max() - rank_matrix.min() + 1)

im = axes[2].imshow(rank_norm, aspect='auto', cmap='RdYlGn_r')
axes[2].set_xticks([0, 1, 2])
axes[2].set_xticklabels(['Granger', 'Correlation', 'Mut. Info'], fontsize=9)
axes[2].set_yticks(range(15))
axes[2].set_yticklabels(top15['feature'].values, fontsize=7)
axes[2].set_title('Top 15 Granger Features\n(green=low rank=high importance)', fontsize=10)
plt.colorbar(im, ax=axes[2], label='Normalised rank')

plt.tight_layout()
plt.savefig('granger_method_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: granger_method_comparison.png')

---

## 6. Directed Information Flow Network

Granger causality is inherently directed — it measures $X \to Y$ (does $X$ predict $Y$?),
not symmetric association. Visualising it as a directed network reveals the information
flow structure among variables.

We draw directed edges from each selected feature to the target, with edge weight
proportional to $-\log(p)$ (heavier edge = stronger evidence of causality).

In [ ]:
# Purpose: Build directed information flow network from Granger results
# Key: Granger causality is directional — this distinguishes it from correlation networks

TARGET_NAME = target.name if target.name else 'Target'
ALPHA_NETWORK = 0.10  # Use relaxed threshold for richer network visualisation

# Build directed graph
G = nx.DiGraph()
G.add_node(TARGET_NAME, node_type='target')

selected_for_network = granger_df[
    granger_df['granger_pvalue'] < ALPHA_NETWORK
].head(20)  # cap at 20 for readability

for _, row in selected_for_network.iterrows():
    feat = row['feature']
    G.add_node(feat, node_type='feature')
    # Edge weight: -log(p) — larger = stronger evidence
    weight = -np.log10(max(row['granger_pvalue'], 1e-10))
    G.add_edge(feat, TARGET_NAME, weight=weight, pvalue=row['granger_pvalue'])

print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} directed edges')
print(f'Features with Granger causality to target (p < {ALPHA_NETWORK}):')
for _, row in selected_for_network.iterrows():
    marker = '*' if row['selected_bh_fdr'] else ' '
    print(f"  [{marker}] {row['feature']}: p={row['granger_pvalue']:.4f}, lag={row['best_lag']}")
print('  * = selected after BH-FDR correction')

In [ ]:
# Purpose: Visualise directed information flow network
# Key: arrows show direction of Granger causation; edge thickness shows strength

fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# Layout: target at centre, features around it
pos = nx.spring_layout(G, seed=42, k=2.5)
# Force target to centre
pos[TARGET_NAME] = np.array([0.0, 0.0])

# Node colours by type
node_colors = []
node_sizes = []
for node in G.nodes():
    if G.nodes[node].get('node_type') == 'target':
        node_colors.append('#E74C3C')  # Red for target
        node_sizes.append(2000)
    elif granger_df[granger_df['feature'] == node]['selected_bh_fdr'].values[0]:
        node_colors.append('#2ECC71')  # Green for BH-selected
        node_sizes.append(800)
    else:
        node_colors.append('#3498DB')  # Blue for marginally significant
        node_sizes.append(500)

# Edge widths proportional to -log(p)
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
max_w = max(edge_weights) if edge_weights else 1
edge_widths = [3 * w / max_w + 0.5 for w in edge_weights]

# Edge colours by p-value
edge_pvals = [G[u][v]['pvalue'] for u, v in G.edges()]
edge_colors = ['#C0392B' if p < 0.01 else '#E67E22' if p < 0.05 else '#F1C40F'
               for p in edge_pvals]

# Draw
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes,
                       alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color=edge_colors,
                       arrows=True, arrowsize=20, connectionstyle='arc3,rad=0.1',
                       ax=ax)

# Legend
from matplotlib.patches import Patch, FancyArrow
legend_elements = [
    Patch(facecolor='#E74C3C', label='Target variable'),
    Patch(facecolor='#2ECC71', label='Selected (BH-FDR)'),
    Patch(facecolor='#3498DB', label='Marginally significant'),
    Patch(facecolor='#C0392B', label='Edge: p<0.01'),
    Patch(facecolor='#E67E22', label='Edge: p<0.05'),
    Patch(facecolor='#F1C40F', label='Edge: p<0.10'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)

ax.set_title(
    f'Directed Information Flow Network: Granger Causality to {TARGET_NAME}\n'
    f'(Arrow thickness ∝ -log(p-value); green nodes pass BH-FDR correction)',
    fontsize=12, fontweight='bold'
)
ax.axis('off')

plt.tight_layout()
plt.savefig('granger_network.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved: granger_network.png')

---

## 7. Summary: Final Feature Rankings

Consolidate results and identify features that rank highly across all three methods.
These "consensus" features are most robustly predictive.

In [ ]:
# Purpose: Build consensus feature ranking across all three methods
# Key: features in the top-10 of all three methods are most reliable

K = min(10, len(comparison))  # Top-K for consensus

top_granger = set(comparison.nsmallest(K, 'granger_rank')['feature'])
top_corr = set(comparison.nsmallest(K, 'corr_rank')['feature'])
top_mi = set(comparison.nsmallest(K, 'mi_rank')['feature'])

consensus_2 = (top_granger & top_corr) | (top_granger & top_mi) | (top_corr & top_mi)
consensus_3 = top_granger & top_corr & top_mi

print('Feature Selection Consensus Analysis')
print('='*55)
print(f'Top-{K} features by each method:')
print(f'  Granger:     {sorted(top_granger)}')
print(f'  Correlation: {sorted(top_corr)}')
print(f'  Mut. Info:   {sorted(top_mi)}')
print(f'\nIn top-{K} of exactly 2 methods: {sorted(consensus_2 - consensus_3)}')
print(f'In top-{K} of all 3 methods:     {sorted(consensus_3)}')
print(f'\nBH-FDR Granger selected: {sorted(granger_df[granger_df["selected_bh_fdr"]]["feature"].tolist())}')

print('\nModule 7 Notebook 1 Complete.')
print('Key findings:')
print(f'  - {n_bh} features pass BH-FDR Granger test at alpha={ALPHA}')
print(f'  - Spearman rank correlation Granger vs Correlation: {rho_granger_corr:.3f}')
print(f'  - Spearman rank correlation Granger vs MI: {rho_granger_mi:.3f}')
print(f'  - {len(consensus_3)} features in consensus top-{K} across all three methods')

---

## Exercise 7.1: Conditional Granger Causality

**Task:** For the top 5 features selected by bivariate Granger, run conditional Granger
causality tests controlling for each other feature in the top 5. Does the ranking change
when you control for common influences?

**Hint:** Build a VAR model with the target + top 5 features, then use `fitted.test_causality()`
to test each feature's contribution while conditioning on the others.

**Expected finding:** Some features that appear to Granger-cause the target bilaterally
may lose significance when you condition on other features — revealing they share a common driver.

In [ ]:
# YOUR CODE HERE
# ---------------
# Step 1: Select the top 5 Granger-selected features
top5_features = granger_df.head(5)['feature'].tolist()
print(f'Top 5 features for conditional test: {top5_features}')

# Step 2: Build VAR model with target + top 5 features
# Hint: from statsmodels.tsa.api import VAR
#       var_data = pd.concat([target, features[top5_features]], axis=1).dropna()
#       model = VAR(var_data)
#       fitted = model.fit(maxlags=4, ic='bic')

# Step 3: Test each feature's conditional Granger causality
# Hint: for feat in top5_features:
#           test = fitted.test_causality(caused=target.name, causing=feat, kind='wald')
#           print(f'{feat}: p={test.pvalue:.4f}')

# TODO: Implement the conditional Granger test above
print('Implement the conditional Granger causality test above.')

In [ ]:
# AUTO-GRADED TEST
def test_exercise_71():
    # Check that top 5 features were identified
    assert len(top5_features) == 5, f'Expected 5 features, got {len(top5_features)}'
    assert all(f in features.columns for f in top5_features), \
        'All top 5 features should be in the features DataFrame'
    print('Exercise 7.1: Top 5 features identified correctly.')
    print('Complete the VAR conditional test implementation to finish this exercise.')

test_exercise_71()

---

## Summary

### Key Takeaways

1. **Granger causality** tests whether past values of a feature improve forecasts of the target beyond the target's own history — it captures **directional predictive information**.

2. **Stationarity first:** run ADF tests and difference/transform non-stationary series before Granger testing.

3. **Multiple testing correction** is mandatory with many features. BH-FDR is preferred over Bonferroni for feature selection because some false positives are acceptable.

4. **Granger, correlation, and MI rankings disagree** substantially — features highly correlated with the target contemporaneously may have little lagged predictive power.

5. **Directed network visualisation** shows the information flow structure, revealing which features are primary predictors vs secondary (driven by the same confounders).

### What's Next

Notebook 02 covers walk-forward feature selection with purged cross-validation — ensuring that the features we select are validated without look-ahead bias.

### References

- McCracken & Ng (2016). FRED-MD. *JBES* 34(4).
- Granger (1969). Investigating Causal Relations. *Econometrica* 37(3).
- Benjamini & Hochberg (1995). Controlling the False Discovery Rate. *JRSS-B* 57(1).